# Phase A v0: first real resynthesis run

A thin driver for `btpvf resynth`. All logic lives in the package: this notebook sets
constants and calls the CLI, nothing else.

**What it produces.** The real condition plus `griffin_lim`, `bigvgan_112m` and `vocos`
for 200 LJSpeech utterances:
- the full-band 22.05 kHz archive and the derived 16 kHz tier;
- `manifest.csv` with its provenance and alignment sidecars;
- the gate reports `gates.json`, `sanity.csv` and `loudness_gain.csv`;
- all of it zipped at the end.

**Gates.** The CLI refuses to start on a blocked or unpinned condition. It then runs:
- the alignment probe (INV-16), per condition;
- manifest validation;
- the analysis band limit on the written audio (INV-17);
- every sanity probe (INV-12).

If any gate fails, no output directory is written and the run cell raises. This notebook
passes `--keep-failed`, so a failed run's partial output is kept aside as
`resynth_v0.failed` for inspection. It is never left at the output path.

**Environment (INV-13): stated, not assumed.** INV-13 says one vocoder per environment,
because vocoder dependency trees conflict. This run deliberately puts `griffin_lim`,
`bigvgan_112m` and `vocos` in the same environment. BigVGAN and Vocos were reported to
coexist in one environment in testing, but that is a reported observation, not something
the repository measured. The CLI allows exactly this set to share a process and refuses
any other combination. The provenance sidecar (`manifest.provenance.json`) records every
package version and the GPU the run used, so the claim can be checked afterwards.

**Out of scope.** `hifigan_v1` and `melgan` stay blocked on manual weight sourcing.

**Needs.**
- GPU accelerator.
- Internet on, for pip and the pinned Hugging Face downloads.
- The LJSpeech dataset attached.

Griffin-Lim runs on CPU and dominates wall-clock time, at roughly 5 s per utterance.
If the repository is private, clone it with a Kaggle Secret; never paste credentials into
a cell (INV-15).

## 1. Constants

In [ ]:
REPO_URL = "https://github.com/SambhavJ2004/btp-vocoder-forensics.git"
REPO_REF = "main"  # pin a commit for a reproducible run; provenance records the commit either way
REPO_DIR = "/kaggle/working/btp-vocoder-forensics"

# LJSpeech root as recorded by the measurement run (data/measurements/meta.json)
LJSPEECH_ROOT = "/kaggle/input/datasets/mathurinache/the-lj-speech-dataset/LJSpeech-1.1"

OUT_DIR = "/kaggle/working/resynth_v0"      # must not exist yet
REPORT_CSV = "/kaggle/working/loudness_gain_report.csv"
N_UTTERANCES = 200
# Neural conditions first: a pin, digest or load failure surfaces before Griffin-Lim's CPU time.
CONDITIONS = ["bigvgan_112m", "vocos", "griffin_lim"]

## 2. Get the package

Clone at `REPO_REF` and install with the `resynth` extra (torch, torchaudio, vocos, huggingface_hub, matplotlib). These are not dependencies of the package itself (INV-13).

In [ ]:
import subprocess
import sys

subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_REF], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[resynth]"], check=True)

## 2b. Optional: Hugging Face token

Anonymous Hugging Face downloads can be rate limited. To authenticate, add a Kaggle Secret named `HF_TOKEN` (Add-ons, then Secrets) and uncomment the two lines below. The token changes nothing about *what* is downloaded, because every download is pinned by revision. The package reads it from the environment only, and never prints, logs or records it (INV-15). Leave the lines commented to run without it.

In [ ]:
import os

# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 3. Checkpoint audit

Shows which conditions are runnable and how each is pinned. `griffin_lim` needs no checkpoint; `bigvgan_112m` is pinned by revision and weight digest; `vocos` by revision.

In [ ]:
subprocess.run(["btpvf", "audit"], check=True, cwd=REPO_DIR)

## 4. Phase A, gated

Generates real plus the three conditions, then runs every gate. `check=True` stops the notebook if any gate fails.

A failed run leaves `resynth_v0.failed` behind (because of `--keep-failed`). The CLI refuses to start while it, `resynth_v0` or `resynth_v0.staging` exists, so remove it deliberately before re-running.

In [ ]:
subprocess.run(
    [
        "btpvf", "resynth",
        "--corpus-root", LJSPEECH_ROOT,
        "--conditions", *CONDITIONS,
        "--out-dir", OUT_DIR,
        "--n-utterances", str(N_UTTERANCES),
        "--keep-failed",
    ],
    check=True,
    cwd=REPO_DIR,
)

## 5. Loudness gain against real

Per condition, paired by utterance, with n and spread:
- `gain_delta_db`, the gain difference;
- `band_offset_db`, the in-band level offset the band-limited primary analysis actually sees.

These closed the loudness-coupling item. Under the band-referenced target (INV-04), `band_offset_db` is expected at the arithmetic residual, ~1e-9 dB, for every condition. The report is written outside the dataset directory, so the dataset is not modified.

In [ ]:
subprocess.run(
    ["btpvf", "loudness", "--manifest", f"{OUT_DIR}/manifest.csv", "--out", REPORT_CSV],
    check=True,
    cwd=REPO_DIR,
)

## 6. Zip the output

Manifest paths are relative to `manifest.csv`, so the zip is self-contained: unzip it anywhere, or mount it at `/kaggle/input/<slug>/`, and every consumer resolves against wherever the manifest now sits. The directory this run generated in is recorded separately, in the provenance sidecar.

In [ ]:
import shutil

shutil.make_archive(OUT_DIR, "zip", root_dir=OUT_DIR)